# Part 3: Data Collection Robot + External Business Insight (Setup → 3.2)
**แหล่งข้อมูล:** TMDB API — `/discover/tv`, `/tv/{id}`, `/tv/{id}/reviews` | **วิธีเก็บ:** API

> This product uses the TMDB API but is not endorsed or certified by TMDB.

## วิธีรัน
**รันทุกเซลล์จากบนลงล่าง** ครั้งแรกจะเรียก API (ประมาณ 5–10 นาที) และบันทึกไฟล์ลง `data_part3/`  
**รอบถัดไป** โค้ดจะพบไฟล์ที่บันทึกไว้และ **อ่านไฟล์เดิมแทนการเรียก API** ข้อมูลจึงไม่เปลี่ยนทุกครั้งที่รัน

| ไฟล์ใน `data_part3/` | คืออะไร |
|---|---|
| `tmdb_tv_raw.jsonl` | ซีรีส์ดิบ (ไม่ซ้ำ id) |
| `tmdb_reviews_raw.jsonl` | รีวิวดิบ |
| `collection_log.json` | จำนวนที่พยายามเก็บ/ล้มเหลว ใช้ในรายงาน 3.2 |
| `tmdb_corpus_clean.jsonl` / `.csv` | **final corpus (1 แถว = 1 รีวิว)** ใช้วิเคราะห์ใน 3.3 และเป็นไฟล์ที่ส่ง |

> ⛔ **ถ้าจะดึงข้อมูลใหม่จริง ๆ** ให้ตั้ง `FORCE_REFETCH = True` ข้อมูลทั้งชุดจะเปลี่ยน (รายชื่อซีรีส์เรียงตามความนิยมรายวัน)  
> ใช้เมื่อรู้ว่าต้องการเท่านั้น และต้องอัปเดตตัวเลขในรายงานทั้งหมด

### 3.1 เลือกแหล่งข้อมูลและวิธีเก็บ  *(แก้ข้อความที่มี ... ให้ตรงกับทีม)*

* **แหล่งข้อมูล:** TMDB (The Movie Database) API v3 — ซีรีส์ทีวีและรีวิวของผู้ใช้
* **คำถาม:** ...
* **ผู้ใช้ผลวิเคราะห์:** ...
* **1 record คือ:** รีวิวของผู้ใช้ 1 ชิ้น (`review_id`) ของซีรีส์ 1 เรื่อง (`doc_id`)
* **วิธีเก็บ:** API (`/discover/tv` → `/tv/{id}` → `/tv/{id}/reviews`)
* **field สำคัญ:** `text`, `user_rating`, `created_at` (รีวิว) | `name`, `genres`, `vote_average`, `vote_count`, `number_of_seasons`, `number_of_episodes` (ซีรีส์)
* **ความรับผิดชอบ:** ใช้ token ของทีม, ใส่ delay, retry เมื่อโดน rate limit (429), **ไม่เก็บชื่อ/username ผู้รีวิว**

In [ ]:
# ===== Setup =====
import os, json, time
from getpass import getpass
import numpy as np
import pandas as pd
import requests

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---- โฟลเดอร์และไฟล์ ----
DATA_DIR      = "./data_part3/"
os.makedirs(DATA_DIR, exist_ok=True)
TV_RAW_PATH   = DATA_DIR + "tmdb_tv_raw.jsonl"
REV_RAW_PATH  = DATA_DIR + "tmdb_reviews_raw.jsonl"
LOG_PATH      = DATA_DIR + "collection_log.json"
CORPUS_JSONL  = DATA_DIR + "tmdb_corpus_clean.jsonl"
CORPUS_CSV    = DATA_DIR + "tmdb_corpus_clean.csv"

# ---- สวิตช์: False = ถ้ามีไฟล์ raw อยู่แล้วให้ใช้ไฟล์เดิม (ข้อมูลไม่เปลี่ยน) ----
FORCE_REFETCH = False

# ---- พารามิเตอร์การเก็บข้อมูล ----
LANGUAGE         = "en-US"
SORT_BY          = "popularity.desc"
MAX_PAGES        = 30        # 1 หน้า = 20 เรื่อง -> ~600 เรื่อง
MAX_REVIEW_PAGES = 30        # 1 หน้า <= 20 รีวิว/เรื่อง (หยุดเองเมื่อหมดหน้า)
DELAY            = 0.1       # วินาที ระหว่าง request

# ---- เกณฑ์ตอน Clean ----
MIN_VOTE_COUNT   = 100       # ซีรีส์ที่มีผู้โหวตน้อยกว่านี้ คะแนนเฉลี่ยยังไม่น่าเชื่อถือ
MIN_REVIEW_CHARS = 50        # รีวิวสั้นกว่านี้ถือว่าใช้วิเคราะห์ไม่ได้

print("ข้อมูลจะถูกบันทึกที่:", os.path.abspath(DATA_DIR))

---
## A. ฟังก์ชันที่ใช้เก็บข้อมูล
ยังไม่เรียก API ในเซลล์นี้ (token จะถูกขอเมื่อมีการเรียกครั้งแรกเท่านั้น)

In [ ]:
# ===== A0: ฟังก์ชัน =====
BASE_URL = "https://api.themoviedb.org/3"
session = None

def get_session():
    """ใช้ "API Read Access Token" (ขึ้นต้น eyJ...) ไม่ใช่ API Key 32 ตัว
    Colab: เก็บเป็น Secret ชื่อ TMDB_TOKEN | ที่อื่น: ตัวแปรสภาพแวดล้อม TMDB_TOKEN หรือพิมพ์ตอนรัน
    ห้ามเขียน token ลงใน notebook"""
    try:
        from google.colab import userdata
        token = userdata.get("TMDB_TOKEN")
    except Exception:
        token = os.environ.get("TMDB_TOKEN") or getpass("วาง TMDB API Read Access Token: ")
    s = requests.Session()
    s.headers.update({"Authorization": f"Bearer {token}", "accept": "application/json"})
    return s

def tmdb_get(path, params=None, retries=4):
    """เรียก TMDB พร้อม retry เมื่อโดน rate limit (429) หรือ server error (5xx)"""
    global session
    if session is None:
        session = get_session()
    for attempt in range(retries):
        r = session.get(BASE_URL + path, params=params, timeout=20)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 2))
            print(f"โดน rate limit รอ {wait}s ...")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"เรียก {path} ไม่สำเร็จหลัง {retries} ครั้ง")

def load_jsonl(path):
    """อ่าน JSON Lines ทีละบรรทัด (ถ้าไฟล์เสียจะบอกเลขบรรทัด)"""
    rows = []
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f"{path} บรรทัด {n} อ่านไม่ได้: {e}")
    return pd.DataFrame(rows)

def save_jsonl(df, path):
    df.to_json(path, orient="records", lines=True, force_ascii=False, date_format="iso")

def load_log():
    return json.load(open(LOG_PATH, encoding="utf-8")) if os.path.exists(LOG_PATH) else {}

def save_log(log):
    json.dump(log, open(LOG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

### A1. เก็บซีรีส์ (`/discover/tv` → `/tv/{id}`)
ถ้ามี `tmdb_tv_raw.jsonl` อยู่แล้วและ `FORCE_REFETCH = False` จะอ่านไฟล์เดิม ไม่เรียก API

In [ ]:
# ===== A1: ซีรีส์ =====
def fetch_discover(max_pages=MAX_PAGES):
    rows = []
    for page in range(1, max_pages + 1):
        data = tmdb_get("/discover/tv", {"language": LANGUAGE, "sort_by": SORT_BY,
                                         "include_adult": "false", "page": page})
        results = data.get("results", [])
        if not results:
            break
        rows.extend(results)
        if page % 10 == 0:
            print(f"ดึงรายการแล้ว {page} หน้า / {len(rows)} เรื่อง")
        if page >= data.get("total_pages", page):
            break
        time.sleep(DELAY)
    return rows

def build_tv_record(d, item, genre_map):
    tv_id = d["id"]
    return {
        "doc_id": str(tv_id),
        "name": d.get("name"),
        "original_name": d.get("original_name"),
        "original_language": d.get("original_language"),
        "origin_country": "|".join(d.get("origin_country") or []),
        "first_air_date": d.get("first_air_date"),
        "last_air_date": d.get("last_air_date"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])) or "|".join(genre_map.get(x, str(x)) for x in item.get("genre_ids", [])),
        "overview": d.get("overview") or item.get("overview") or "",
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "popularity": d.get("popularity"),
        "number_of_seasons": d.get("number_of_seasons"),
        "number_of_episodes": d.get("number_of_episodes"),
        "status": d.get("status"),
        "in_production": d.get("in_production"),
        "source": "tmdb",
        "url": f"https://www.themoviedb.org/tv/{tv_id}",
    }

if os.path.exists(TV_RAW_PATH) and not FORCE_REFETCH:
    print("พบ", TV_RAW_PATH, "-> ใช้ไฟล์เดิม ไม่เรียก API")
    df_tv_raw = load_jsonl(TV_RAW_PATH)
    df_tv_raw["doc_id"] = df_tv_raw["doc_id"].astype(str)
    log = load_log()
else:
    genre_map = {g["id"]: g["name"] for g in tmdb_get("/genre/tv/list", {"language": LANGUAGE})["genres"]}
    list_rows = fetch_discover()

    seen, uniq = set(), []                         # /discover/tv อาจส่งซีรีส์เดิมซ้ำข้ามหน้า -> ตัดก่อนเรียก detail
    for it in list_rows:
        if it["id"] not in seen:
            seen.add(it["id"]); uniq.append(it)

    records, failed = [], []
    for i, item in enumerate(uniq, start=1):
        try:
            records.append(build_tv_record(tmdb_get(f"/tv/{item['id']}", {"language": LANGUAGE}), item, genre_map))
        except Exception as e:
            failed.append((item["id"], str(e)))
        if i % 100 == 0:
            print(f"detail {i}/{len(uniq)}")
        time.sleep(DELAY)

    df_tv_raw = pd.DataFrame(records)
    save_jsonl(df_tv_raw, TV_RAW_PATH)
    log = {"series_listed": len(list_rows), "series_list_duplicates": len(list_rows) - len(uniq),
           "series_detail_ok": len(records), "series_detail_failed": len(failed)}
    save_log(log)
    print(f"✅ ซีรีส์: รายการ {len(list_rows)} -> ไม่ซ้ำ {len(uniq)} -> detail สำเร็จ {len(records)} / ล้มเหลว {len(failed)}")

print("ซีรีส์ดิบ:", len(df_tv_raw), "เรื่อง")

### A2. เก็บรีวิว (`/tv/{id}/reviews`)
วนทีละซีรีส์ **ไม่ซ้ำ id** (ไม่วนตามตารางที่มี id ซ้ำ) และไม่เก็บชื่อผู้รีวิว

In [ ]:
# ===== A2: รีวิว =====
MAX_REVIEW_PAGES_LOCAL = MAX_REVIEW_PAGES

def fetch_reviews(tv_id, max_pages=MAX_REVIEW_PAGES_LOCAL):
    out = []
    for page in range(1, max_pages + 1):
        data = tmdb_get(f"/tv/{tv_id}/reviews", {"page": page})
        for r in data.get("results", []):
            out.append({
                "review_id": r.get("id"),
                "doc_id": str(tv_id),
                "text": r.get("content") or "",
                "user_rating": (r.get("author_details") or {}).get("rating"),   # อาจเป็น None
                "created_at": r.get("created_at"),
                # ไม่เก็บ author/username เพื่อลดข้อมูลส่วนบุคคล (PDPA)
            })
        if page >= data.get("total_pages", 1):
            break
        time.sleep(DELAY)
    return out

if os.path.exists(REV_RAW_PATH) and not FORCE_REFETCH:
    print("พบ", REV_RAW_PATH, "-> ใช้ไฟล์เดิม ไม่เรียก API")
    df_rev_raw = load_jsonl(REV_RAW_PATH)
    df_rev_raw["doc_id"] = df_rev_raw["doc_id"].astype(str)
else:
    ids = df_tv_raw["doc_id"].drop_duplicates().tolist()
    rev_rows, rev_failed = [], []
    for i, tv_id in enumerate(ids, start=1):
        try:
            rev_rows.extend(fetch_reviews(tv_id))
        except Exception as e:
            rev_failed.append(tv_id)
        if i % 100 == 0:
            print(f"{i}/{len(ids)} เรื่อง | รีวิวสะสม {len(rev_rows)}")
        time.sleep(DELAY)

    n_first_fail = len(rev_failed)                  # ลองใหม่อีก 1 รอบสำหรับเรื่องที่ล้มเหลว
    still_failed = []
    for tv_id in rev_failed:
        try:
            rev_rows.extend(fetch_reviews(tv_id))
        except Exception:
            still_failed.append(tv_id)
        time.sleep(DELAY)

    df_rev_raw = pd.DataFrame(rev_rows)
    save_jsonl(df_rev_raw, REV_RAW_PATH)
    log.update({"review_series_requested": len(ids), "review_series_failed_first_try": n_first_fail,
                "review_series_failed_final": len(still_failed), "review_rows_raw": len(df_rev_raw)})
    save_log(log)
    print(f"✅ รีวิว: ขอจาก {len(ids)} เรื่อง -> ได้ {len(df_rev_raw)} แถว | ล้มเหลวรอบแรก {n_first_fail} / หลังลองใหม่ {len(still_failed)}")

print("รีวิวดิบ:", len(df_rev_raw), "แถว")

---
### 3.2 ตรวจคุณภาพข้อมูล (Clean + Quality Report)

ลำดับ: **Clean ซีรีส์ → Clean รีวิว → Join → รายงาน → บันทึก final corpus**  
ทุกเซลล์ในส่วนนี้ใช้เฉพาะไฟล์ raw ที่โหลดแล้ว จึงรันซ้ำได้โดยผลเดิม

In [ ]:
# ===== 3.2-1: Clean ซีรีส์ =====
tv = df_tv_raw.copy()
tv_rows_raw = len(tv)

n_tv_dup = tv.duplicated("doc_id").sum();               tv = tv.drop_duplicates("doc_id")
n_tv_noid = (tv["doc_id"].isna() | tv["doc_id"].astype(str).str.strip().eq("")).sum()
tv = tv[~(tv["doc_id"].isna() | tv["doc_id"].astype(str).str.strip().eq(""))]

tv["first_air_date"] = pd.to_datetime(tv["first_air_date"], errors="coerce")
tv["last_air_date"]  = pd.to_datetime(tv["last_air_date"],  errors="coerce")
n_tv_nodate = tv["first_air_date"].isna().sum()         # รายงานไว้ ไม่ตัด (ใช้รีวิวเป็นหลัก)

n_tv_lowvote = (tv["vote_count"] < MIN_VOTE_COUNT).sum()
tv = tv[tv["vote_count"] >= MIN_VOTE_COUNT]

tv_clean = tv.reset_index(drop=True)
print("===== TV SERIES CLEANING =====")
print(f"ซีรีส์ดิบ                        : {tv_rows_raw}")
print(f"id ซ้ำ / ไม่มี id                : {n_tv_dup} / {n_tv_noid}")
print(f"ไม่มี first_air_date (ไม่ตัด)    : {n_tv_nodate}")
print(f"ตัด vote_count < {MIN_VOTE_COUNT}              : {n_tv_lowvote}")
print(f"ซีรีส์หลัง Clean                 : {len(tv_clean)}")

In [ ]:
# ===== 3.2-2: Clean รีวิว + Join กับซีรีส์ =====
rev = df_rev_raw.copy()
rev_rows_raw = len(rev)

n_rev_dup = rev.duplicated("review_id").sum();          rev = rev.drop_duplicates("review_id")

rev["text"] = rev["text"].fillna("").str.strip()
n_rev_empty = (rev["text"] == "").sum();                rev = rev[rev["text"] != ""]

rev["text_length"] = rev["text"].str.len()
n_rev_short = (rev["text_length"] < MIN_REVIEW_CHARS).sum();   rev = rev[rev["text_length"] >= MIN_REVIEW_CHARS]

n_rev_duptext = rev.duplicated("text").sum();           rev = rev.drop_duplicates("text")

rev["created_at"] = pd.to_datetime(rev["created_at"], errors="coerce", utc=True)
n_rev_nodate = rev["created_at"].isna().sum();          rev = rev.dropna(subset=["created_at"])

n_rev_noseries = (~rev["doc_id"].isin(tv_clean["doc_id"])).sum()   # รีวิวของซีรีส์ที่ถูกตัดในขั้นก่อน
corp = rev.merge(tv_clean, on="doc_id", how="inner").reset_index(drop=True)

print("===== REVIEW CLEANING =====")
print(f"รีวิวดิบ                          : {rev_rows_raw}")
print(f"review_id ซ้ำ                     : {n_rev_dup}")
print(f"ข้อความว่าง                       : {n_rev_empty}")
print(f"สั้นกว่า {MIN_REVIEW_CHARS} ตัวอักษร              : {n_rev_short}")
print(f"ข้อความซ้ำ                        : {n_rev_duptext}")
print(f"ไม่มีวันที่                       : {n_rev_nodate}")
print(f"ซีรีส์ถูกตัด (vote_count ต่ำ)     : {n_rev_noseries}")
print(f"Final corpus                      : {len(corp)}")

In [ ]:
# ===== 3.2-3: Quality Report + บันทึก final corpus =====
print("===== DATA QUALITY REPORT =====")
print("[ซีรีส์]")
print(f"  รายการที่ดึง / ซ้ำข้ามหน้า      : {log.get('series_listed', 'n/a')} / {log.get('series_list_duplicates', 'n/a')}")
print(f"  detail สำเร็จ / ล้มเหลว          : {log.get('series_detail_ok', len(df_tv_raw))} / {log.get('series_detail_failed', 'n/a')}")
print(f"  หลัง Clean (vote_count >= {MIN_VOTE_COUNT})  : {len(tv_clean)}")
print("[รีวิว]")
print(f"  ล้มเหลวหลังลองใหม่ (ระดับซีรีส์)  : {log.get('review_series_failed_final', 'n/a')}")
print(f"  เก็บได้ -> final corpus           : {rev_rows_raw} -> {len(corp)}")

print("\n===== FINAL CORPUS =====")
per_show = corp["doc_id"].value_counts()
print(f"รีวิว {len(corp)} | ซีรีส์ที่มีรีวิว {corp['doc_id'].nunique()} เรื่อง | รีวิวต่อเรื่อง median {per_show.median():.0f}, max {per_show.max()}")
print(f"10 เรื่องที่มีรีวิวมากสุด = {per_show.head(10).sum() / len(corp):.1%} ของรีวิวทั้งหมด")
print(f"ช่วงเวลารีวิว: {corp['created_at'].min():%Y-%m-%d} ถึง {corp['created_at'].max():%Y-%m-%d}")
print(f"มี user_rating: {corp['user_rating'].notna().sum()} / {len(corp)} ({corp['user_rating'].notna().mean():.0%})")
print("\nภาษาต้นฉบับ (สัดส่วนรีวิว):")
print(corp["original_language"].value_counts(normalize=True).head(5).round(3).to_string())
genres_covered = sorted({g for gs in corp["genres"].dropna() for g in gs.split("|") if g})
print(f"\nGenres ที่ครอบคลุม ({len(genres_covered)}):", genres_covered)
print("\nmissing ในคอลัมน์สำคัญ:")
print(corp[["text", "created_at", "name", "genres", "vote_average", "vote_count"]].isna().sum().to_string())

print("\nผ่านเกณฑ์ >= 200 valid analytical records:", "✅" if len(corp) >= 200 else "❌ ไม่ถึง 200 -> เพิ่ม MAX_PAGES แล้วตั้ง FORCE_REFETCH = True")

# ---- บันทึก final corpus ----
save_jsonl(corp, CORPUS_JSONL)
corp.to_csv(CORPUS_CSV, index=False, encoding="utf-8-sig")
check = load_jsonl(CORPUS_JSONL)
print(f"\nบันทึกแล้ว: {CORPUS_JSONL} ({len(check)} แถว) และ {CORPUS_CSV}")

**สรุปคุณภาพข้อมูล (3.2)** *(กรอกตัวเลขจากผลรันด้านบน)*

* พยายามเก็บ ... ซีรีส์ (ซ้ำข้ามหน้า ... ) → รีวิวดิบ ... แถว
* ตัดรีวิวซ้ำ ... / ว่าง ... / สั้นกว่า 50 ตัวอักษร ... / ข้อความซ้ำ ... / ซีรีส์ vote_count < 100 ...
* **Final corpus ... รีวิว** จาก ... ซีรีส์ ครอบคลุมปี ... – ...
* **ข้อจำกัด:** ซีรีส์ถูกเลือกจาก `/discover/tv` เรียงตามความนิยม ณ วันที่ดึง ซีรีส์ที่ไม่มีรีวิวไม่ปรากฏใน corpus และรีวิวกระจุกอยู่ที่ไม่กี่เรื่อง